# Colab: teacher Q&A then QLoRA of Qwen3.5-2B

Run top to bottom on a free T4. Writes under Drive (`thesis-kd/`) so a disconnect loses at most the current row. Re-run the same cells to resume.

This is response-level distillation: `qwen3.5:9b` writes grounded Q&A, then LoRA-SFT on the 2B. Not soft-target / logit KD.

**Before you start**

1. Runtime → Change runtime type → T4 GPU.
2. On the laptop, rebuild the index (default sources have no Ubuntu Server docs) and copy `chunks.jsonl` or `kd/seed_chunks.jsonl` to Drive `thesis-kd/`.
3. Write the held-out test questions first (`your-heldout.jsonl`). `build_sft.py --holdout` needs them so train rows do not leak the test set. Do not use `eval/test.jsonl`.
4. Default seed mix (no Server docs): tldr=460, command-manpages=295, man-pages=115, gnome-user-docs=230, ubuntu-docs=29.


## Teacher path: Ollama, not transformers 4-bit

`kd/teacher_generate.py` and `kd/judge_rows.py` only call Ollama at `localhost:11434`. Ollama's `qwen3.5:9b` is already Q4 (~6.6–7.6 GB) and fits a T4 16 GB card. A transformers 4-bit teacher would be a second stack and would not reuse those scripts.

Set `USE_4B_TEACHER = True` if the 9B pull or load fails. That is a weaker teacher, not the same run. Stop Ollama before SFT so the 2B has the GPU.

The last cell prints the Modelfile and the laptop command `ollama create thesis-student -f Modelfile`.


In [ ]:
# Flags. Edit these, then run the rest in order.
REPO_URL = "https://github.com/Birdabo404/sui-SLM.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis-kd"
REPO_DIR = "/content/sui-SLM"
HELDOUT = ""  # empty: Drive your-heldout.jsonl, then repo eval/your-heldout.jsonl

USE_4B_TEACHER = False  # True -> qwen3.5:4b
USE_QLORA = True        # Unsloth prefers bf16 LoRA on Qwen3.5; set False if 4-bit looks bad
DRY_RUN = False         # True: skip Ollama pull, teacher, and GPU SFT
TEACHER_LIMIT = 0       # 0 = all remaining seeds; 20 is a smoke pass
SAVE_EVERY = 10
EPOCHS = 1
MAX_SEQ = 2048
LORA_R = 16
BATCH = 1
GRAD_ACCUM = 8

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

work = Path(DRIVE_ROOT)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not Colab; using DRIVE_ROOT as a normal folder.")
    work.mkdir(parents=True, exist_ok=True)

work.mkdir(parents=True, exist_ok=True)
os.chdir(work)
print("Python", sys.version.split()[0])
print("work", work)

In [ ]:
repo = Path(REPO_DIR)
if not (repo / "kd" / "teacher_generate.py").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo)])
sys.path.insert(0, str(repo))
os.chdir(repo)
print("repo", repo)
print("head", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
pkgs = [
    "sentence-transformers==3.3.1",
    "datasets",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
if not DRY_RUN:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--no-cache-dir", "unsloth", "unsloth_zoo",
    ])
print("pip ok")

In [ ]:
from kd.colab_helpers import (
    OLLAMA_NAME,
    STUDENT_HF,
    count_jsonl,
    ollama_create_cmd,
    questions_txt_from_jsonl,
    require_seeds,
    resolve_chunks,
    resolve_heldout,
    resolve_seeds,
    teacher_ollama_name,
    write_modelfile,
)

heldout = resolve_heldout(
    [HELDOUT, work / "your-heldout.jsonl", repo / "eval" / "your-heldout.jsonl"],
    work,
)
holdout_txt = work / "holdout_questions.txt"
n_hold = questions_txt_from_jsonl(heldout, holdout_txt)
print("heldout", heldout, "questions", n_hold)

In [ ]:
seeds = resolve_seeds(work, repo)
if seeds is None:
    chunks = resolve_chunks(work, repo)
    if chunks is None:
        require_seeds(work, repo)
    seeds = work / "seed_chunks.jsonl"
    subprocess.check_call([
        sys.executable, str(repo / "kd" / "make_seed_chunks.py"),
        "--chunks", str(chunks),
        "--out", str(seeds),
    ])
elif seeds.resolve() != (work / "seed_chunks.jsonl").resolve():
    shutil.copy2(seeds, work / "seed_chunks.jsonl")
    seeds = work / "seed_chunks.jsonl"
print("seeds", seeds, "rows", count_jsonl(seeds))

In [ ]:
teacher = teacher_ollama_name(USE_4B_TEACHER)
print("teacher", teacher, "(4B fallback)" if USE_4B_TEACHER else "(9B)")
if DRY_RUN:
    print("DRY_RUN: skip Ollama install/pull")
else:
    if shutil.which("ollama") is None:
        subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])
    log = work / "ollama.log"
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open(log, "ab"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    import time
    import urllib.request
    for i in range(30):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(2)
    else:
        raise SystemExit("Ollama did not start. See " + str(log))
    subprocess.check_call(["ollama", "pull", teacher])

In [ ]:
raw = work / "teacher_raw.jsonl"
cmd = [
    sys.executable, str(repo / "kd" / "teacher_generate.py"),
    "--seeds", str(seeds),
    "--out", str(raw),
    "--model", teacher,
]
if TEACHER_LIMIT:
    cmd.extend(["--limit", str(TEACHER_LIMIT)])
if DRY_RUN:
    print("DRY_RUN: skip teacher_generate.", "already", count_jsonl(raw))
else:
    print("resume teacher; already", count_jsonl(raw), "flush every row, status every", SAVE_EVERY)
    subprocess.check_call(cmd)
print("teacher_raw", raw, "rows", count_jsonl(raw))

In [ ]:
judged = work / "teacher_judged.jsonl"
if DRY_RUN:
    print("DRY_RUN: skip judge_rows.", "already", count_jsonl(judged))
else:
    subprocess.check_call([
        sys.executable, str(repo / "kd" / "judge_rows.py"),
        "--raw", str(raw),
        "--seeds", str(seeds),
        "--out", str(judged),
        "--model", teacher,
    ])
print("teacher_judged", judged, "rows", count_jsonl(judged))

In [ ]:
sft_in = judged if judged.is_file() and count_jsonl(judged) else raw
train_path = work / "sft_train.jsonl"
val_path = work / "sft_val.jsonl"
if not sft_in.is_file() or count_jsonl(sft_in) == 0:
    raise SystemExit("No teacher rows to build SFT from: " + str(sft_in))
subprocess.check_call([
    sys.executable, str(repo / "kd" / "build_sft.py"),
    "--raw", str(sft_in),
    "--seeds", str(seeds),
    "--holdout", str(holdout_txt),
    "--out-train", str(train_path),
    "--out-val", str(val_path),
])
print("train", count_jsonl(train_path), "val", count_jsonl(val_path))

In [ ]:
# Free the GPU before SFT. Teacher and student should not share the T4.
if not DRY_RUN and shutil.which("ollama"):
    subprocess.run(["pkill", "-f", "ollama"], check=False)
    print("stopped Ollama")

In [ ]:
# QLoRA / LoRA SFT of Qwen3.5-2B. Small epochs. Loss is printed and written to Drive.
if DRY_RUN:
    print("DRY_RUN: skip GPU SFT. train rows", count_jsonl(train_path))
    if train_path.is_file() and count_jsonl(train_path):
        import json
        row = json.loads(next(l for l in open(train_path, encoding="utf-8") if l.strip()))
        print("sample roles", [m["role"] for m in row["messages"]])
else:
    if USE_QLORA:
        print("USE_QLORA=True. Unsloth reports large 4-bit error on Qwen3.5; switch to False for bf16 LoRA (~5 GB).")
    try:
        from unsloth import FastLanguageModel
        use_unsloth = True
    except ImportError:
        use_unsloth = False
        print("unsloth missing; peft + bitsandbytes fallback")

    from datasets import load_dataset
    from trl import SFTConfig, SFTTrainer

    data = load_dataset("json", data_files={"train": str(train_path), "val": str(val_path)})
    out_dir = work / "sft-out"
    if use_unsloth:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=STUDENT_HF,
            max_seq_length=MAX_SEQ,
            load_in_4bit=USE_QLORA,
            load_in_16bit=not USE_QLORA,
            full_finetuning=False,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=LORA_R,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
            max_seq_length=MAX_SEQ,
        )
    else:
        import torch
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        tok_kwargs = {"trust_remote_code": True}
        tokenizer = AutoTokenizer.from_pretrained(STUDENT_HF, **tok_kwargs)
        if USE_QLORA:
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            model = AutoModelForCausalLM.from_pretrained(
                STUDENT_HF, quantization_config=bnb, device_map="auto", trust_remote_code=True,
            )
            model = prepare_model_for_kbit_training(model)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                STUDENT_HF, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
            )
        model = get_peft_model(model, LoraConfig(
            r=LORA_R, lora_alpha=LORA_R, lora_dropout=0,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none", task_type="CAUSAL_LM",
        ))

    args = SFTConfig(
        output_dir=str(out_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=2e-4,
        logging_steps=5,
        save_steps=50,
        max_seq_length=MAX_SEQ,
        optim="adamw_8bit",
        seed=3407,
        report_to=[],
        dataset_kwargs={"skip_prepare_dataset": False},
    )
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=data["train"],
        eval_dataset=data["val"] if count_jsonl(val_path) else None,
        args=args,
    )
    result = trainer.train()
    trainer.save_model(str(work / "lora"))
    tokenizer.save_pretrained(str(work / "lora"))
    loss_path = work / "sft_loss.json"
    hist = result.metrics
    loss_path.write_text(__import__("json").dumps(hist, indent=2), encoding="utf-8")
    print("loss metrics", hist)
    print("wrote", loss_path)

In [ ]:
gguf_dir = work / "gguf"
gguf_dir.mkdir(parents=True, exist_ok=True)
if DRY_RUN:
    print("DRY_RUN: skip merge/GGUF")
else:
    try:
        from unsloth import FastLanguageModel
        model.save_pretrained_merged(str(work / "merged"), tokenizer, save_method="merged_16bit")
        model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method="q4_k_m")
    except Exception as e:
        print("unsloth export failed:", e)
        print("Install llama.cpp and convert the merged folder, then quantize to Q4_K_M.")
        raise
    print("gguf dir", list(gguf_dir.glob("*"))[:20])

In [ ]:
ggufs = sorted(gguf_dir.glob("*.gguf"))
gguf_name = ggufs[0].name if ggufs else "qwen35-2b-student-q4_k_m.gguf"
mf = write_modelfile(work / "Modelfile", gguf_name)
print(mf.read_text())
print()
print("On the laptop, copy the GGUF and Modelfile into one folder, then:")
print("  " + ollama_create_cmd(OLLAMA_NAME))
print("Eval:")
print(
    "  python eval/run_eval.py --split test --no-sweep "
    "--questions eval/your-heldout.jsonl "
    "--out logs/eval-test.csv --distilled " + OLLAMA_NAME
)
print("USE_4B_TEACHER", USE_4B_TEACHER, "USE_QLORA", USE_QLORA, "DRY_RUN", DRY_RUN)